In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from pathlib import Path
import kagglehub

print("Скачиваем датасет...")
path = kagglehub.dataset_download("atulanandjha/temperature-readings-iot-devices")

csv_files = list(Path(path).rglob("*.csv"))
if csv_files:
    csv_path = str(csv_files[0])
    print(f"Найден файл: {csv_path}")
    
    df = pd.read_csv(csv_path)
    print(f"Размер датасета: {df.shape}")
    print(f"Поля: {df.columns.tolist()}")
    
    df.to_csv('temperature_dataset.csv', index=False)
    print("Датасет сохранен как 'temperature_dataset.csv'")

Скачиваем датасет...
Найден файл: /Users/zhmikhail/.cache/kagglehub/datasets/atulanandjha/temperature-readings-iot-devices/versions/1/IOT-temp.csv
Размер датасета: (97606, 5)
Колонки: ['id', 'room_id/id', 'noted_date', 'temp', 'out/in']
Датасет сохранен как 'temperature_dataset.csv'


In [ ]:
# Дело наше начинается с неспешной загрузки датасета — этого безгласного архива, таящего в себе истории, ещё не рассказанные. 
# Мы вверяем машине эти цифровые фолианты, дабы извлечь из них сокровенный смысл и холодную, неопровержимую истину.

df = pd.read_csv('temperature_dataset.csv')

# Помацаем что у нас тут
print("=== ИНФОРМАЦИЯ О ДАТАСЕТЕ ===")
print(f"Размер: {df.shape}")
print(f"Колонки: {df.columns.tolist()}")
print(f"\nТипы данных:")
print(df.dtypes)
print(f"\nПервые 5 строк:")
print(df.head())
print(f"\nПропущенные значения:")
print(df.isnull().sum())

=== ИНФОРМАЦИЯ О ДАТАСЕТЕ ===
Размер: (97606, 5)
Колонки: ['id', 'room_id/id', 'noted_date', 'temp', 'out/in']

Типы данных:
id            object
room_id/id    object
noted_date    object
temp           int64
out/in        object
dtype: object

Первые 5 строк:
                                    id  room_id/id        noted_date  temp  \
0  __export__.temp_log_196134_bd201015  Room Admin  08-12-2018 09:30    29   
1  __export__.temp_log_196131_7bca51bc  Room Admin  08-12-2018 09:30    29   
2  __export__.temp_log_196127_522915e3  Room Admin  08-12-2018 09:29    41   
3  __export__.temp_log_196128_be0919cf  Room Admin  08-12-2018 09:29    41   
4  __export__.temp_log_196126_d30b72fb  Room Admin  08-12-2018 09:29    31   

  out/in  
0     In  
1     In  
2    Out  
3    Out  
4     In  

Пропущенные значения:
id            0
room_id/id    0
noted_date    0
temp          0
out/in        0
dtype: int64


In [40]:
# Стандартизируем названия колонок
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]
df.columns = [col.strip().lower().replace('/', '_') for col in df.columns]
print(f"Стандартизированные колонки: {df.columns.tolist()}")

# Переименуем для удобства
column_mapping = {}
for col in df.columns:
    if 'date' in col:
        column_mapping[col] = 'noted_date'
    elif 'temp' in col or 'temperature' in col:
        column_mapping[col] = 'temp'
    elif 'out' in col or 'in' in col:
        column_mapping[col] = 'out_in'

if column_mapping:
    df = df.rename(columns=column_mapping)
    print(f"После переименования: {df.columns.tolist()}")

# Преобразуем дату
if 'noted_date' in df.columns:
    df['noted_date'] = pd.to_datetime(df['noted_date'], errors='coerce')
    df = df.dropna(subset=['noted_date'])
    print(f"Диапазон дат: {df['noted_date'].min()} - {df['noted_date'].max()}")

Стандартизированные колонки: ['id', 'room_id_id', 'noted_date', 'temp', 'out_in']
После переименования: ['id', 'room_id_id', 'noted_date', 'temp', 'out_in']
Диапазон дат: 2018-01-11 00:06:00 - 2018-12-10 23:55:00


In [ ]:
def create_tables():
    """Создание таблиц в PostgreSQL"""
    conn = get_connection()
    if not conn:
        return
    
    cursor = conn.cursor()
    
    # Таблица для полной загрузки (исторические данные)
    create_full_table = """
    CREATE TABLE IF NOT EXISTS temperature_full (
        id SERIAL PRIMARY KEY,
        noted_date TIMESTAMP,
        temperature FLOAT,
        location VARCHAR(10),
        device_id VARCHAR(50),
        load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        batch_id VARCHAR(50)
    );
    """
    
    # Таблица для инкрементальной загрузки (все равно это не прод, а домашка - решил разделить для простоты проверки)
    create_incremental_table = """
    CREATE TABLE IF NOT EXISTS temperature_incremental (
        id SERIAL PRIMARY KEY,
        noted_date TIMESTAMP,
        temperature FLOAT,
        location VARCHAR(10),
        device_id VARCHAR(50),
        load_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        batch_id VARCHAR(50),
        UNIQUE(noted_date, device_id)  -- Уникальность по дате и устройству
    );
    """
    
    # Таблица для отслеживания загрузок
    create_load_history = """
    CREATE TABLE IF NOT EXISTS load_history (
        id SERIAL PRIMARY KEY,
        load_type VARCHAR(20),
        batch_id VARCHAR(50),
        start_date TIMESTAMP,
        end_date TIMESTAMP,
        records_loaded INTEGER,
        load_status VARCHAR(20),
        error_message TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    
    try:
        cursor.execute(create_full_table)
        cursor.execute(create_incremental_table)
        cursor.execute(create_load_history)
        conn.commit()
        print("✓ Таблицы созданы успешно")
        
        # Проверяем существование таблиц
        cursor.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
        AND table_name IN ('temperature_full', 'temperature_incremental', 'load_history')
        """)
        tables = cursor.fetchall()
        print(f"✓ Созданы таблицы: {[t[0] for t in tables]}")
        
    except Exception as e:
        print(f"✗ Ошибка при создании таблиц: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()

# Создаем таблицы
create_tables()

✓ Подключение к PostgreSQL установлено
✓ Таблицы созданы успешно
✓ Созданы таблицы: ['load_history', 'temperature_full', 'temperature_incremental']


In [52]:
def full_load_to_postgres(df, batch_size=1000):
    """
    Выполняет полную загрузку всех данных в таблицу temperature_full.
    Очищает целевую таблицу и загружает все записи из DataFrame заново.
    Записывает метаданные о загрузке в таблицу load_history.
    
    Параметры:
        df: DataFrame с данными (должен содержать колонки noted_date, temp, out_in, id)
        batch_size: размер пачки для загрузки (по умолчанию 1000)
    
    Возвращает:
        Количество успешно загруженных записей
    """
    print("=== ПОЛНАЯ ЗАГРУЗКА (FULL LOAD) ===")
    
    conn = get_connection()
    if not conn:
        return 0
    
    cursor = conn.cursor()
    
    batch_id = f"full_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    try:
        cursor.execute("""
        INSERT INTO load_history (load_type, batch_id, start_date, load_status)
        VALUES (%s, %s, %s, %s)
        """, ('full', batch_id, datetime.now(), 'started'))
        conn.commit()
    except Exception as e:
        print(f"Ошибка записи в историю: {e}")
    
    loaded_rows = 0
    
    try:
        cursor.execute("TRUNCATE TABLE temperature_full")
        print("Таблица temperature_full очищена")
        
        records_to_insert = []
        
        for idx, row in df.iterrows():
            try:
                noted_date = row['noted_date']
                if pd.isna(noted_date):
                    continue
                
                temperature = row['temp']
                if pd.isna(temperature):
                    temperature = None
                
                location = str(row['out_in']).strip()[:10] if pd.notna(row['out_in']) else 'unknown'
                
                device_id = 'unknown'
                if pd.notna(row['id']):
                    id_str = str(row['id'])
                    if '_' in id_str:
                        device_id = f"device_{id_str.split('_')[-1][:10]}"
                    else:
                        device_id = f"device_{id_str[:10]}"
                elif 'room_id/id' in df.columns and pd.notna(row['room_id/id']):
                    device_id = f"room_{str(row['room_id/id'])[:10]}"
                else:
                    device_id = f"default_{idx}"
                
                records_to_insert.append((
                    noted_date,
                    temperature,
                    location,
                    device_id,
                    batch_id
                ))
                
            except Exception as e:
                continue
        
        print(f"Подготовлено {len(records_to_insert)} записей из {len(df)}")
        
        total_rows = len(records_to_insert)
        
        for i in range(0, total_rows, batch_size):
            batch = records_to_insert[i:i+batch_size]
            
            if batch:
                insert_query = """
                INSERT INTO temperature_full 
                (noted_date, temperature, location, device_id, batch_id)
                VALUES %s
                """
                execute_values(cursor, insert_query, batch)
                loaded_rows += len(batch)
            
            if i % (batch_size * 10) == 0 or i + batch_size >= total_rows:
                print(f"Загружено {min(i+batch_size, total_rows)} из {total_rows} записей")
        
        conn.commit()
        print(f"Полная загрузка завершена: {loaded_rows} записей")
        
        cursor.execute("""
        UPDATE load_history 
        SET end_date = %s, records_loaded = %s, load_status = %s
        WHERE batch_id = %s
        """, (datetime.now(), loaded_rows, 'success', batch_id))
        conn.commit()
        
        cursor.execute("SELECT COUNT(*) FROM temperature_full")
        total_in_db = cursor.fetchone()[0]
        print(f"В таблице temperature_full теперь: {total_in_db} записей")
        
        cursor.execute("""
        SELECT 
            MIN(noted_date), 
            MAX(noted_date),
            COUNT(DISTINCT device_id),
            AVG(temperature)
        FROM temperature_full
        """)
        min_date, max_date, device_count, avg_temp = cursor.fetchone()
        print(f"Период данных: {min_date} - {max_date}")
        print(f"Уникальных устройств: {device_count}")
        if avg_temp is not None:
            print(f"Средняя температура: {avg_temp:.2f}°C")
        
    except Exception as e:
        print(f"Ошибка при полной загрузке: {e}")
        
        try:
            cursor.execute("""
            UPDATE load_history 
            SET end_date = %s, load_status = %s, error_message = %s
            WHERE batch_id = %s
            """, (datetime.now(), 'failed', str(e)[:500], batch_id))
            conn.commit()
        except:
            pass
        conn.rollback()
    finally:
        cursor.close()
        conn.close()
    
    return loaded_rows


def incremental_load_to_postgres(df, last_n_days=7, batch_size=500):
    """
    Выполняет инкрементальную загрузку данных за последние N дней.
    Загружает данные за указанный период от максимальной даты в DataFrame.
    Использует UPSERT: обновляет существующие записи и добавляет новые.
    Удаляет дубликаты по комбинации (noted_date, device_id).
    Записывает метаданные о загрузке в таблицу load_history.
    
    Параметры:
        df: DataFrame с данными (должен содержать колонку noted_date)
        last_n_days: количество дней для загрузки от максимальной даты (по умолчанию 7)
        batch_size: размер пачки для загрузки (по умолчанию 500)
    
    Возвращает:
        Количество успешно загруженных/обновленных записей
    """
    print(f"=== ИНКРЕМЕНТАЛЬНАЯ ЗАГРУЗКА (последние {last_n_days} дней) ===")
    
    conn = get_connection()
    if not conn:
        return 0
    
    cursor = conn.cursor()
    
    batch_id = f"inc_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    try:
        cursor.execute("""
        INSERT INTO load_history (load_type, batch_id, start_date, load_status)
        VALUES (%s, %s, %s, %s)
        """, ('incremental', batch_id, datetime.now(), 'started'))
        conn.commit()
    except Exception as e:
        print(f"Ошибка записи в историю: {e}")
    
    total_loaded = 0
    errors = 0
    
    try:
        if 'noted_date' in df.columns:
            if not pd.api.types.is_datetime64_any_dtype(df['noted_date']):
                df = df.copy()
                df['noted_date'] = pd.to_datetime(df['noted_date'], errors='coerce')
            
            max_date = df['noted_date'].max()
            
            if pd.isna(max_date):
                print("Не удалось определить максимальную дату")
                cursor.execute("""
                UPDATE load_history 
                SET end_date = %s, records_loaded = %s, load_status = %s
                WHERE batch_id = %s
                """, (datetime.now(), 0, 'failed_no_date', batch_id))
                conn.commit()
                return 0
            
            cutoff_date = max_date - timedelta(days=last_n_days)
            
            print(f"Максимальная дата в данных: {max_date}")
            print(f"Берем данные с: {cutoff_date}")
            print(f"Период загрузки: {last_n_days} дней ({cutoff_date.date()} - {max_date.date()})")
            
            mask = (df['noted_date'] >= cutoff_date) & (df['noted_date'] <= max_date)
            df_filtered = df[mask].copy()
            
            print(f"Данных за последние {last_n_days} дней: {len(df_filtered)} записей")
            
            if len(df_filtered) == 0:
                print("Нет данных для указанного периода")
                cursor.execute("""
                UPDATE load_history 
                SET end_date = %s, records_loaded = %s, load_status = %s
                WHERE batch_id = %s
                """, (datetime.now(), 0, 'no_data', batch_id))
                conn.commit()
                return 0
        else:
            print("Нет колонки с датой в данных")
            cursor.execute("""
            UPDATE load_history 
            SET end_date = %s, records_loaded = %s, load_status = %s
            WHERE batch_id = %s
            """, (datetime.now(), 0, 'failed_no_date_col', batch_id))
            conn.commit()
            return 0
        
        records_to_insert = []
        
        for idx, row in df_filtered.iterrows():
            try:
                noted_date = row['noted_date']
                if pd.isna(noted_date):
                    continue
                
                temperature = None
                if 'temp' in row and pd.notna(row['temp']):
                    try:
                        temperature = float(row['temp'])
                    except:
                        temperature = None
                
                location = 'unknown'
                if 'out_in' in row and pd.notna(row['out_in']):
                    location = str(row['out_in']).strip()[:10]
                elif 'out_in' in row and pd.notna(row['out_in']):
                    location = str(row['out_in']).strip()[:10]
                
                device_id = 'unknown'
                if 'id' in row and pd.notna(row['id']):
                    id_val = row['id']
                    if isinstance(id_val, str):
                        parts = id_val.split('_')
                        if len(parts) > 1:
                            device_id = f"dev_{parts[-1][:15]}"
                        else:
                            device_id = f"dev_{id_val[:15]}"
                    else:
                        device_id = f"dev_{str(id_val)[:15]}"
                else:
                    device_id = f"row_{idx}"
                
                records_to_insert.append((
                    noted_date,
                    temperature,
                    location,
                    device_id,
                    batch_id
                ))
                
            except Exception as e:
                errors += 1
                continue
        
        print(f"Подготовлено {len(records_to_insert)} записей")
        if errors > 0:
            print(f"Пропущено {errors} записей с ошибками")
        
        if not records_to_insert:
            print("Нет валидных записей для загрузки")
            cursor.execute("""
            UPDATE load_history 
            SET end_date = %s, records_loaded = %s, load_status = %s
            WHERE batch_id = %s
            """, (datetime.now(), 0, 'no_valid_data', batch_id))
            conn.commit()
            return 0
        
        dates = sorted(set(r[0].date() for r in records_to_insert))
        print(f"Будет загружено {len(dates)} дней: {dates[0]} - {dates[-1]}")
        
        unique_records = {}
        for record in records_to_insert:
            key = (record[0], record[3])
            unique_records[key] = record
        
        print(f"После удаления дубликатов: {len(unique_records)} уникальных записей")
        
        unique_records_list = list(unique_records.values())
        
        loaded_in_batch = 0
        batch_errors = 0
        
        for i in range(0, len(unique_records_list), batch_size):
            batch = unique_records_list[i:i+batch_size]
            
            try:
                for record in batch:
                    try:
                        check_query = """
                        SELECT COUNT(*) 
                        FROM temperature_incremental 
                        WHERE noted_date = %s AND device_id = %s
                        """
                        cursor.execute(check_query, (record[0], record[3]))
                        exists = cursor.fetchone()[0] > 0
                        
                        if exists:
                            update_query = """
                            UPDATE temperature_incremental 
                            SET temperature = %s, 
                                location = %s, 
                                load_timestamp = CURRENT_TIMESTAMP,
                                batch_id = %s
                            WHERE noted_date = %s AND device_id = %s
                            """
                            cursor.execute(update_query, 
                                         (record[1], record[2], record[4], record[0], record[3]))
                        else:
                            insert_query = """
                            INSERT INTO temperature_incremental 
                            (noted_date, temperature, location, device_id, batch_id)
                            VALUES (%s, %s, %s, %s, %s)
                            """
                            cursor.execute(insert_query, record)
                        
                        total_loaded += 1
                        
                    except Exception as record_error:
                        batch_errors += 1
                        try:
                            simple_insert = """
                            INSERT INTO temperature_incremental 
                            (noted_date, temperature, location, device_id, batch_id)
                            VALUES (%s, %s, %s, %s, %s)
                            ON CONFLICT (noted_date, device_id) DO NOTHING
                            """
                            cursor.execute(simple_insert, record)
                            total_loaded += 1
                            batch_errors -= 1
                        except:
                            continue
                
                conn.commit()
                loaded_in_batch += len(batch)
                
                if (i // batch_size) % 5 == 0 or i + batch_size >= len(unique_records_list):
                    print(f"Загружено {min(i+batch_size, len(unique_records_list))} из {len(unique_records_list)} записей")
                    
            except Exception as batch_error:
                print(f"Ошибка пачки: {batch_error}")
                conn.rollback()
                for record in batch:
                    try:
                        simple_insert = """
                        INSERT INTO temperature_incremental 
                        (noted_date, temperature, location, device_id, batch_id)
                        VALUES (%s, %s, %s, %s, %s)
                        ON CONFLICT (noted_date, device_id) DO NOTHING
                        """
                        cursor.execute(simple_insert, record)
                        total_loaded += 1
                    except:
                        batch_errors += 1
                        continue
                conn.commit()
        
        print(f"Инкрементальная загрузка завершена")
        print(f"Успешно: {total_loaded} записей")
        if batch_errors > 0:
            print(f"Ошибок: {batch_errors} записей")
        
        status = 'success' if batch_errors == 0 else 'partial'
        cursor.execute("""
        UPDATE load_history 
        SET end_date = %s, records_loaded = %s, load_status = %s
        WHERE batch_id = %s
        """, (datetime.now(), total_loaded, status, batch_id))
        conn.commit()
        
        cursor.execute("SELECT COUNT(*) FROM temperature_incremental")
        total_in_db = cursor.fetchone()[0]
        print(f"В таблице temperature_incremental теперь: {total_in_db} записей")
        
        cursor.execute("""
        SELECT 
            DATE(noted_date) as load_date, 
            COUNT(*) as records
        FROM temperature_incremental 
        WHERE batch_id = %s
        GROUP BY DATE(noted_date)
        ORDER BY load_date DESC
        LIMIT 10
        """, (batch_id,))
        
        print("Загруженные дни:")
        results = cursor.fetchall()
        for date, count in results:
            print(f"  {date}: {count} записей")
        
        if results:
            cursor.execute("""
            SELECT 
                AVG(temperature) as avg_temp,
                MIN(temperature) as min_temp,
                MAX(temperature) as max_temp
            FROM temperature_incremental 
            WHERE batch_id = %s
            """, (batch_id,))
            
            avg_temp, min_temp, max_temp = cursor.fetchone()
            if avg_temp is not None:
                print(f"Статистика температуры:")
                print(f"  Средняя: {avg_temp:.1f}°C")
                print(f"  Минимальная: {min_temp:.1f}°C")
                print(f"  Максимальная: {max_temp:.1f}°C")
        
    except Exception as e:
        print(f"Критическая ошибка при инкрементальной загрузке: {e}")
        
        try:
            cursor.execute("""
            UPDATE load_history 
            SET end_date = %s, load_status = %s, error_message = %s
            WHERE batch_id = %s
            """, (datetime.now(), 'failed', str(e)[:500], batch_id))
            conn.commit()
        except:
            pass
        
        conn.rollback()
    finally:
        cursor.close()
        conn.close()
    
    return total_loaded

In [ ]:
# Пересоздаем таблицы
recreate_tables()

print("="*70)
print("ТЕСТ 1: ПОЛНАЯ ЗАГРУЗКА ВСЕХ ДАННЫХ")
print("="*70)
full_count = full_load_to_postgres(df)
print(f"\n✅ Загружено в FULL таблицу: {full_count:,} записей")

print("\n" + "="*70)
print("="*70)
print("ПРОСТОЙ ТЕСТ ИНКРЕМЕНТАЛЬНОЙ ЗАГРУЗКИ")
print("="*70)

# Берем небольшую часть данных для теста
df_sample = df.sample(n=1000, random_state=42).copy()
print(f"Тестовый датасет: {len(df_sample)} записей")

# Тест 1: Загрузка последних 3 дней
print("\n--- Тест 1: Последние 3 дня ---")
inc1 = incremental_load_to_postgres(df_sample, last_n_days=3)
print(f"Загружено: {inc1} записей")

# Тест 2: Загрузка последних 7 дней (другие данные)
print("\n--- Тест 2: Последние 7 дней (другие данные) ---")
df_sample2 = df.sample(n=2000, random_state=123).copy()
inc2 = incremental_load_to_postgres(df_sample2, last_n_days=7)
print(f"Загружено: {inc2} записей")

# Тест 3: Попробуем загрузить те же данные еще раз (должно обновиться) ((и даже обновляется))
print("\n--- Тест 3: Повторная загрузка тех же данных (обновление) ---")
inc3 = incremental_load_to_postgres(df_sample, last_n_days=3)
print(f"Обновлено/добавлено: {inc3} записей")

print("\n" + "="*70)
print("ПРОВЕРКА РЕЗУЛЬТАТОВ")
print("="*70)

def quick_check():
    conn = get_connection()
    if not conn:
        return
    
    cursor = conn.cursor()
    
    try:
        print("\n📊 Быстрая проверка:")
        
        cursor.execute("SELECT COUNT(*) FROM temperature_incremental")
        total = cursor.fetchone()[0]
        print(f"Всего в incremental: {total} записей")
        
        cursor.execute("SELECT COUNT(DISTINCT DATE(noted_date)) FROM temperature_incremental")
        days = cursor.fetchone()[0]
        print(f"Уникальных дней: {days}")
        
        cursor.execute("SELECT COUNT(DISTINCT device_id) FROM temperature_incremental")
        devices = cursor.fetchone()[0]
        print(f"Уникальных устройств: {devices}")
        
        # Проверяем дубликаты
        cursor.execute("""
        SELECT COUNT(*) as duplicate_count
        FROM (
            SELECT noted_date, device_id, COUNT(*)
            FROM temperature_incremental
            GROUP BY noted_date, device_id
            HAVING COUNT(*) > 1
        ) as duplicates
        """)
        dup_count = cursor.fetchone()[0]
        
        if dup_count == 0:
            print("✅ Нет дубликатов (noted_date + device_id уникальны)")
        else:
            print(f"⚠ Найдено {dup_count} дубликатов")
        
        # Последние загрузки
        print("\n📋 Последние 3 загрузки:")
        cursor.execute("""
        SELECT load_type, batch_id, records_loaded, load_status
        FROM load_history
        ORDER BY created_at DESC
        LIMIT 3
        """)
        
        for load_type, batch_id, records, status in cursor.fetchall():
            print(f"  {batch_id[:12]}... ({load_type}): {records} записей, статус: {status}")
        
    except Exception as e:
        print(f"Ошибка проверки: {e}")
    finally:
        cursor.close()
        conn.close()

quick_check()

✓ Подключение к PostgreSQL установлено
✓ Таблицы пересозданы
ТЕСТ 1: ПОЛНАЯ ЗАГРУЗКА ВСЕХ ДАННЫХ
=== ПОЛНАЯ ЗАГРУЗКА (FULL LOAD) ===
✓ Подключение к PostgreSQL установлено
Таблица temperature_full очищена
Подготовлено 49944 записей из 49944
Загружено 1000 из 49944 записей
Загружено 11000 из 49944 записей
Загружено 21000 из 49944 записей
Загружено 31000 из 49944 записей
Загружено 41000 из 49944 записей
Загружено 49944 из 49944 записей
Полная загрузка завершена: 49944 записей
В таблице temperature_full теперь: 49944 записей
Период данных: 2018-01-11 00:06:00 - 2018-12-10 23:55:00
Уникальных устройств: 49943
Средняя температура: 31.38°C

✅ Загружено в FULL таблицу: 49,944 записей

ПРОСТОЙ ТЕСТ ИНКРЕМЕНТАЛЬНОЙ ЗАГРУЗКИ
Тестовый датасет: 1000 записей

--- Тест 1: Последние 3 дня ---
=== ИНКРЕМЕНТАЛЬНАЯ ЗАГРУЗКА (последние 3 дней) ===
✓ Подключение к PostgreSQL установлено
Максимальная дата в данных: 2018-12-10 23:27:00
Берем данные с: 2018-12-07 23:27:00
Период загрузки: 3 дней (2018-12-07 